In [1]:
# Step 1: uninstall conflicting versions
!pip uninstall -y torch torchvision torchaudio

# Step 2: install compatible PyTorch (IMPORTANT)
!pip install -q torch==2.3.1+cu118 torchvision==0.18.1+cu118 \
    --index-url https://download.pytorch.org/whl/cu118

# Step 3: install rest
!pip install -q grad-cam
!pip install -q git+https://github.com/openai/CLIP.git
!pip install -q ftfy regex

import torch
print("Torch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))
print("Available:", torch.cuda.is_available())

Found existing installation: torch 2.3.1+cu118
Uninstalling torch-2.3.1+cu118:
  Successfully uninstalled torch-2.3.1+cu118
Found existing installation: torchvision 0.18.1+cu118
Uninstalling torchvision-0.18.1+cu118:
  Successfully uninstalled torchvision-0.18.1+cu118
  Preparing metadata (setup.py) ... done
Torch: 2.3.1+cu118
CUDA: 11.8
GPU: Tesla P100-PCIE-16GB
Available: True


In [2]:
# Step 1: Install only the specialized libraries you need
# Kaggle already has a perfectly compatible torch/torchvision version pre-installed.
!pip install -q grad-cam ftfy regex
!pip install -q git+https://github.com/openai/CLIP.git

# Step 2: Verify the environment
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Version: {torch.version.cuda}")
print(f"Device: {torch.cuda.get_device_name(0)}")

# Step 3: Check if the 'sm' (compute capability) is supported
# This list must include the architecture of your GPU (e.g., 'sm_75' for T4 or 'sm_60' for P100)
print(f"Supported Architectures: {torch.cuda.get_arch_list()}")

  Preparing metadata (setup.py) ... done
PyTorch Version: 2.3.1+cu118
CUDA Version: 11.8
Device: Tesla P100-PCIE-16GB
Supported Architectures: ['sm_50', 'sm_60', 'sm_70', 'sm_75', 'sm_80', 'sm_86', 'sm_37', 'sm_90']


In [ ]:
# Cell 2 — Clone the paper's repo
import os

!git clone https://github.com/tomburgert/feature-reliance.git
os.chdir('feature-reliance')

# See what's in there
!find . -name "*.py" | head -30
!cat README.md

In [ ]:
# Cell 3 — Create folders
!mkdir -p contribution/models
!mkdir -p contribution/logs  

In [ ]:
%%writefile contribution/suppression_utils.py

import numpy as np
import torch
import cv2
import random


def tensor_to_numpy(img_tensor):
    return img_tensor.permute(1, 2, 0).numpy().astype(np.float32)


def numpy_to_tensor(img_np):
    return torch.from_numpy(img_np).permute(2, 0, 1)


def patch_shuffle_tensor(img_tensor, grid_size=6):
    img = tensor_to_numpy(img_tensor)
    H, W, C = img.shape
    ph, pw = H // grid_size, W // grid_size
    patches = [
        img[i*ph:(i+1)*ph, j*pw:(j+1)*pw, :].copy()
        for i in range(grid_size) for j in range(grid_size)
    ]
    random.shuffle(patches)
    result = np.zeros_like(img)
    for idx, (i, j) in enumerate(
        [(i, j) for i in range(grid_size) for j in range(grid_size)]
    ):
        result[i*ph:(i+1)*ph, j*pw:(j+1)*pw, :] = patches[idx]
    return numpy_to_tensor(result)


def bilateral_filter_tensor(img_tensor, d=11,
                             sigma_color=170, sigma_space=75):
    img = tensor_to_numpy(img_tensor)
    img_uint8 = (img * 255).clip(0, 255).astype(np.uint8)
    filtered = cv2.bilateralFilter(img_uint8, d, sigma_color, sigma_space)
    return numpy_to_tensor(filtered.astype(np.float32) / 255.0)


def grayscale_tensor(img_tensor):
    img = tensor_to_numpy(img_tensor)
    gray = (0.299 * img[:,:,0] +
            0.587 * img[:,:,1] +
            0.114 * img[:,:,2])
    gray_3ch = np.stack([gray, gray, gray], axis=2).astype(np.float32)
    return numpy_to_tensor(gray_3ch)


def apply_suppression(img_tensor, mode, grid_size=6):
    if mode == 'normal':
        return img_tensor
    elif mode == 'shape':
        return patch_shuffle_tensor(img_tensor, grid_size)
    elif mode == 'texture':
        return bilateral_filter_tensor(img_tensor)
    elif mode == 'color':
        return grayscale_tensor(img_tensor)
    else:
        raise ValueError(f"Unknown mode: {mode}")

In [ ]:
%%writefile contribution/curriculum_dataset.py

import random
import torch
from torch.utils.data import Dataset
import sys
sys.path.append('/kaggle/working/contribution')
from suppression_utils import apply_suppression


def get_curriculum_probs(epoch: int, total_epochs: int) -> dict:
    """
    Dynamic schedule: starts easy (mostly normal images),
    ramps up suppression difficulty as training progresses.

    Epoch 1  → normal=0.70, shape=0.10, texture=0.10, color=0.10
    Epoch T  → normal=0.10, shape=0.30, texture=0.30, color=0.30
    """
    progress = (epoch - 1) / max(total_epochs - 1, 1)  # 0.0 → 1.0
    p_normal  = 0.70 - 0.60 * progress          # 0.70 → 0.10
    p_shared  = (1.0 - p_normal) / 3            # equal split for the rest
    return {
        'normal' : round(p_normal,  4),
        'shape'  : round(p_shared,  4),
        'texture': round(p_shared,  4),
        'color'  : round(p_shared,  4),
    }


class CurriculumDataset(Dataset):
    def __init__(self, base_dataset, grid_size=6, normalize=None):
        self.dataset   = base_dataset
        self.modes     = ['normal', 'shape', 'texture', 'color']
        self.weights   = [0.70, 0.10, 0.10, 0.10]   # epoch 1 defaults
        self.grid_size = grid_size
        self.normalize = normalize

    def set_epoch(self, epoch: int, total_epochs: int):
        """Call at the start of each epoch to update the mixing schedule."""
        probs = get_curriculum_probs(epoch, total_epochs)
        self.weights = [
            probs['normal'],
            probs['shape'],
            probs['texture'],
            probs['color'],
        ]

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        img, label = self.dataset[idx]
        mode = random.choices(self.modes, weights=self.weights)[0]
        img  = apply_suppression(img, mode, self.grid_size)
        if self.normalize:
            img = self.normalize(img)
        return img, label, mode

In [ ]:
%%writefile contribution/train.py

import sys, os, argparse, json
sys.path.append('/kaggle/working/contribution')

import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as T
from torchvision.datasets import CIFAR10
from torch.utils.data import DataLoader
from curriculum_dataset import CurriculumDataset, get_curriculum_probs
from tqdm import tqdm

CIFAR_MEAN = [0.4914, 0.4822, 0.4465]
CIFAR_STD  = [0.2023, 0.1994, 0.2010]


def get_loaders(mode='standard', batch_size=128,
                data_root='/kaggle/working/data'):
    normalize = T.Normalize(mean=CIFAR_MEAN, std=CIFAR_STD)

    val_transform = T.Compose([T.Resize((32, 32)), T.ToTensor(), normalize])
    val_set = CIFAR10(root=data_root, train=False, download=True,
                      transform=val_transform)
    val_loader = DataLoader(val_set, batch_size=batch_size,
                            shuffle=False, num_workers=2, pin_memory=True)

    if mode == 'standard':
        train_transform = T.Compose([
            T.RandomHorizontalFlip(),
            T.RandomCrop(32, padding=4),
            T.ToTensor(),
            normalize,
        ])
        train_set = CIFAR10(root=data_root, train=True, download=True,
                            transform=train_transform)
        train_loader = DataLoader(train_set, batch_size=batch_size,
                                  shuffle=True, num_workers=2, pin_memory=True)
        return train_loader, val_loader, None

    elif mode == 'curriculum':
        base_transform = T.Compose([
            T.RandomHorizontalFlip(),
            T.RandomCrop(32, padding=4),
            T.ToTensor(),
        ])
        raw_train = CIFAR10(root=data_root, train=True, download=True,
                            transform=base_transform)
        curriculum_set = CurriculumDataset(
            base_dataset=raw_train,
            normalize=normalize,
        )
        train_loader = DataLoader(curriculum_set, batch_size=batch_size,
                                  shuffle=True, num_workers=2, pin_memory=True)
        return train_loader, val_loader, curriculum_set


def evaluate(model, loader, device):
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for batch in loader:
            imgs, labels = batch[0].to(device), batch[1].to(device)
            correct += model(imgs).argmax(1).eq(labels).sum().item()
            total   += labels.size(0)
    return correct / total


def train(mode='standard', epochs=30, batch_size=128, save_path=None):

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    print(f"\n{'═' * 80}")
    print(f"{' ' * 25}TRAINING STARTED - {mode.upper()} MODE")
    print(f"{'═' * 80}")
    print(f"  Epochs      : {epochs}")
    print(f"  Batch size  : {batch_size}")
    print(f"  Device      : {device}")
    
    if mode == 'curriculum':
        p0 = get_curriculum_probs(1, epochs)
        pT = get_curriculum_probs(epochs, epochs)
        print(f"  Curriculum  : Starts → normal={p0['normal']:.0%} | "
              f"shape={p0['shape']:.0%} | texture={p0['texture']:.0%} | color={p0['color']:.0%}")
        print(f"                Ends   → normal={pT['normal']:.0%} | "
              f"shape={pT['shape']:.0%} | texture={pT['texture']:.0%} | color={pT['color']:.0%}")
    
    print(f"{'═' * 80}\n")

    model = models.resnet18(weights=None, num_classes=10).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)
    criterion = nn.CrossEntropyLoss()

    train_loader, val_loader, curriculum_set = get_loaders(mode, batch_size)
    history = []

    for epoch in range(1, epochs + 1):
        # Update curriculum schedule
        if mode == 'curriculum' and curriculum_set is not None:
            curriculum_set.set_epoch(epoch, epochs)
            probs = get_curriculum_probs(epoch, epochs)

        model.train()
        correct = total = 0
        running_loss = 0.0
        mode_counts = {'normal': 0, 'shape': 0, 'texture': 0, 'color': 0}

        print(f"\n Epoch {epoch:02d}/{epochs} {'─' * 60}")

        pbar = tqdm(train_loader, desc="  Training", 
                    leave=True, dynamic_ncols=True, ncols=100)

        for batch in pbar:
            imgs = batch[0].to(device)
            labels = batch[1].to(device)
            
            if len(batch) == 3 and mode == 'curriculum':
                for m in batch[2]:
                    mode_counts[m] += 1

            optimizer.zero_grad()
            out = model(imgs)
            loss = criterion(out, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            running_loss += loss.item()
            correct += out.argmax(1).eq(labels).sum().item()
            total += labels.size(0)

            pbar.set_postfix({
                'loss': f'{running_loss / (pbar.n + 1):.4f}',
                'acc': f'{correct / total:.4f}'
            })

        scheduler.step()

        train_acc = correct / total
        avg_loss = running_loss / len(train_loader)
        val_acc = evaluate(model, val_loader, device)

        history.append({
            'epoch': epoch,
            'train_loss': round(avg_loss, 4),
            'train_acc': round(train_acc, 4),
            'val_acc': round(val_acc, 4),
        })

        # Final epoch summary
        print(f"  → Loss: {avg_loss:.4f} | Train Acc: {train_acc:.4f} | "
              f"Val Acc: {val_acc:.4f} | LR: {scheduler.get_last_lr()[0]:.2e}")

        if mode == 'curriculum':
            total_s = sum(mode_counts.values())
            if total_s > 0:
                actual = " | ".join(f"{k}={v/total_s:.1%}" for k, v in mode_counts.items())
                scheduled = " | ".join(f"{k}={probs[k]:.1%}" for k in mode_counts)
                print(f"  → Actual distribution    : {actual}")
                print(f"  → Scheduled distribution : {scheduled}")

        print(f"{'─' * 80}")

    # Save model
    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        torch.save({
            'model_state_dict': model.state_dict(),
            'history': history,
            'mode': mode,
        }, save_path)
        print(f"\n Model successfully saved → {save_path}")

    print(f"\n Training completed successfully: {mode.upper()} mode\n")
    return model, history

In [ ]:
!pip install tqdm


In [3]:
# Run training directly in notebook (easier than subprocess on Kaggle)
import sys
sys.path.append('/kaggle/working/contribution')

# Import your train function
from train import train

print("="*80)
print("STARTING TRAINING PIPELINE")
print("="*80)

# Train standard model
print("\n TRAINING STANDARD MODEL...")
std_model, std_history = train(
    mode='standard',
    epochs=30,
    batch_size=128,
    save_path='/kaggle/working/contribution/models/standard.pt'
)

# Train curriculum model
print("\n TRAINING CURRICULUM MODEL...")
cur_model, cur_history = train(
    mode='curriculum',
    epochs=30,
    batch_size=128,
    save_path='/kaggle/working/contribution/models/curriculum.pt'
)

print("="*80)
print("ALL TRAINING COMPLETED SUCCESSFULLY!")
print("="*80)

STARTING TRAINING PIPELINE

 TRAINING STANDARD MODEL...

════════════════════════════════════════════════════════════════════════════════
                         TRAINING STARTED - STANDARD MODE
════════════════════════════════════════════════════════════════════════════════
  Epochs      : 30
  Batch size  : 128
  Device      : cuda
════════════════════════════════════════════════════════════════════════════════

Files already downloaded and verified
Files already downloaded and verified

 Epoch 01/30 ────────────────────────────────────────────────────────────


  Training: 100%|██████████| 391/391 [00:25<00:00, 15.52it/s, loss=1.5415, acc=0.4392]


  → Loss: 1.5376 | Train Acc: 0.4392 | Val Acc: 0.5395 | LR: 9.97e-04
────────────────────────────────────────────────────────────────────────────────

 Epoch 02/30 ────────────────────────────────────────────────────────────


  Training: 100%|██████████| 391/391 [00:24<00:00, 15.89it/s, loss=1.1634, acc=0.5859]


  → Loss: 1.1604 | Train Acc: 0.5859 | Val Acc: 0.6288 | LR: 9.89e-04
────────────────────────────────────────────────────────────────────────────────

 Epoch 03/30 ────────────────────────────────────────────────────────────


...

  → Loss: 0.8494 | Train Acc: 0.6951 | Val Acc: 0.8287 | LR: 1.19e-05
  → Actual distribution    : normal=14.2% | shape=28.9% | texture=28.3% | color=28.6%
  → Scheduled distribution : normal=14.1% | shape=28.6% | texture=28.6% | color=28.6%
────────────────────────────────────────────────────────────────────────────────

 Epoch 29/30 ────────────────────────────────────────────────────────────


  Training: 100%|██████████| 391/391 [00:25<00:00, 15.53it/s, loss=0.8563, acc=0.6925]


  → Loss: 0.8541 | Train Acc: 0.6925 | Val Acc: 0.8280 | LR: 3.74e-06
  → Actual distribution    : normal=12.3% | shape=29.2% | texture=29.4% | color=29.1%
  → Scheduled distribution : normal=12.1% | shape=29.3% | texture=29.3% | color=29.3%
────────────────────────────────────────────────────────────────────────────────

 Epoch 30/30 ────────────────────────────────────────────────────────────


  Training: 100%|██████████| 391/391 [00:25<00:00, 15.52it/s, loss=0.8615, acc=0.6892]


  → Loss: 0.8593 | Train Acc: 0.6892 | Val Acc: 0.8282 | LR: 1.00e-06
  → Actual distribution    : normal=9.9% | shape=30.1% | texture=29.9% | color=30.0%
  → Scheduled distribution : normal=10.0% | shape=30.0% | texture=30.0% | color=30.0%
────────────────────────────────────────────────────────────────────────────────

 Model successfully saved → /kaggle/working/contribution/models/curriculum.pt

 Training completed successfully: CURRICULUM mode

ALL TRAINING COMPLETED SUCCESSFULLY!


In [9]:
import json, sys
sys.path.append('/kaggle/working/contribution')

import torch
import torchvision.transforms as T
import torchvision.models as models
from torchvision.datasets import CIFAR10
from contribution.suppression_utils import apply_suppression

CIFAR_MEAN = [0.4914, 0.4822, 0.4465]
CIFAR_STD  = [0.2023, 0.1994, 0.2010]
device = torch.device('cuda')


def load_model(path):
    m = models.resnet18(weights=None, num_classes=10).to(device)
    ckpt = torch.load(path, map_location=device)
    m.load_state_dict(ckpt['model_state_dict'])
    m.eval()
    return m


def eval_suppression(model, dataset, mode, grid_size=6):
    normalize = T.Normalize(CIFAR_MEAN, CIFAR_STD)
    correct = total = 0
    with torch.no_grad():
        for idx in range(len(dataset)):
            img, label = dataset[idx]
            sup = normalize(
                apply_suppression(img, mode, grid_size)
            ).unsqueeze(0).to(device)
            pred = model(sup).argmax(1).item()
            correct += (pred == label)
            total   += 1
    return correct / total


raw_val = CIFAR10(
    root='/kaggle/working/data', train=False, download=True,
    transform=T.Compose([T.Resize((32, 32)), T.ToTensor()]),
)

std_model = load_model(
    '/kaggle/working/contribution/models/standard.pt')
cur_model = load_model(
    '/kaggle/working/contribution/models/curriculum.pt')

# ── Evaluation conditions ─────────────────────────────────────────────────
conditions = {
    'original'     : ('normal',  6),
    'local_shape'  : ('shape',   6),
    'global_shape' : ('shape',   3),
    'texture'      : ('texture', 6),
    'color'        : ('color',   6),
}

# ── Run eval ──────────────────────────────────────────────────────────────
accs = {}
for cond, (mode, grid) in conditions.items():
    accs[cond] = {
        'standard'  : eval_suppression(std_model, raw_val, mode, grid),
        'curriculum': eval_suppression(cur_model, raw_val, mode, grid),
    }

orig_std = accs['original']['standard']
orig_cur = accs['original']['curriculum']

# ── Feature reliance scores: drop = original_acc - suppressed_acc ─────────
# A large drop means the model *depended* on that feature cue.
# A small drop means the model is robust when that cue is removed.
reliance = {}
for cond in ['local_shape', 'global_shape', 'texture', 'color']:
    reliance[cond] = {
        'standard'  : round(orig_std - accs[cond]['standard'],   4),
        'curriculum': round(orig_cur - accs[cond]['curriculum'],  4),
    }

# ── Print accuracy table ──────────────────────────────────────────────────
print(f"\n{'─'*65}")
print(f"  SUPPRESSION ACCURACY")
print(f"{'─'*65}")
print(f"  {'Condition':<20} {'Standard':>10} {'Curriculum':>12} {'Δ':>10}")
print(f"  {'─'*58}")
for cond in conditions:
    s = accs[cond]['standard']
    c = accs[cond]['curriculum']
    d = c - s
    arrow = "↑" if d > 0 else ("↓" if d < 0 else " ")
    print(f"  {cond:<20} {s:>10.4f} {c:>12.4f} "
          f"  {arrow}{abs(d):.4f}")

# ── Print reliance score table ────────────────────────────────────────────
print(f"\n{'─'*65}")
print(f"  FEATURE RELIANCE SCORES  (original_acc − suppressed_acc)")
print(f"  Higher = more dependent on that feature cue")
print(f"{'─'*65}")
print(f"  {'Feature':<20} {'Standard':>10} {'Curriculum':>12} {'Δ reliance':>12}")
print(f"  {'─'*58}")
for cond, r in reliance.items():
    d = r['curriculum'] - r['standard']
    arrow = "↑" if d > 0 else ("↓" if d < 0 else " ")
    print(f"  {cond:<20} {r['standard']:>10.4f} {r['curriculum']:>12.4f}"
          f"  {arrow}{abs(d):.4f}")

# ── Save results ──────────────────────────────────────────────────────────
output = {
    'accuracy' : accs,
    'reliance' : reliance,
    'baseline_accs': {
        'standard_original'  : orig_std,
        'curriculum_original': orig_cur,
    },
}
save_path = ('/kaggle/working/contribution/'
             'logs/reliance_comparison.json')
with open(save_path, 'w') as f:
    json.dump(output, f, indent=2)

print(f"\nResults saved → {save_path}")

Files already downloaded and verified

─────────────────────────────────────────────────────────────────
  SUPPRESSION ACCURACY
─────────────────────────────────────────────────────────────────
  Condition              Standard   Curriculum          Δ
  ──────────────────────────────────────────────────────────
  original                 0.8484       0.8282   ↓0.0202
  local_shape              0.2293       0.4218   ↑0.1925
  global_shape             0.3248       0.4135   ↑0.0887
  texture                  0.5883       0.7779   ↑0.1896
  color                    0.7540       0.7888   ↑0.0348

─────────────────────────────────────────────────────────────────
  FEATURE RELIANCE SCORES  (original_acc − suppressed_acc)
  Higher = more dependent on that feature cue
─────────────────────────────────────────────────────────────────
  Feature                Standard   Curriculum   Δ reliance
  ──────────────────────────────────────────────────────────
  local_shape              0.6191       0.4

In [1]:
!zip -r 1_contribution.zip /kaggle/working/contribution
!zip -r 2_contribution.zip /kaggle/working/data

  adding: kaggle/working/contribution/ (stored 0%)
  adding: kaggle/working/contribution/__pycache__/ (stored 0%)
  adding: kaggle/working/contribution/__pycache__/curriculum_dataset.cpython-312.pyc (deflated 44%)
  adding: kaggle/working/contribution/__pycache__/train.cpython-312.pyc (deflated 50%)
  adding: kaggle/working/contribution/__pycache__/gradcam_viz.cpython-312.pyc (deflated 43%)
  adding: kaggle/working/contribution/__pycache__/suppression_utils.cpython-312.pyc (deflated 44%)
  adding: kaggle/working/contribution/models/ (stored 0%)
  adding: kaggle/working/contribution/models/standard.pt (deflated 7%)
  adding: kaggle/working/contribution/models/curriculum.pt (deflated 7%)
  adding: kaggle/working/contribution/gradcam_viz.py (deflated 58%)
  adding: kaggle/working/contribution/train.py (deflated 68%)
  adding: kaggle/working/contribution/curriculum_dataset.py (deflated 61%)
  adding: kaggle/working/contribution/logs/ (stored 0%)
  adding: kaggle/working/contribution/logs/r